In [7]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.models as models
from torchvision.models import ResNet18_Weights  # resnet18 weights API is documented. [web:41]


In [8]:
proc = Path("data/processed")

X_train_tab = np.load(proc / "X_train_tab.npy")
X_val_tab   = np.load(proc / "X_val_tab.npy")
X_test_tab  = np.load(proc / "X_test_tab.npy")

y_train = np.load(proc / "y_train.npy")
y_val   = np.load(proc / "y_val.npy")

train_ids = pd.read_csv(proc / "train_ids.csv")["id"].astype(str).values
val_ids   = pd.read_csv(proc / "val_ids.csv")["id"].astype(str).values
test_ids  = pd.read_csv(proc / "test_ids.csv")["id"].astype(str).values

scaler_df = pd.read_csv(proc / "tabular_scaler.csv")
TAB_FEATURES = scaler_df["feature"].tolist()

X_train_tab.shape, X_val_tab.shape, X_test_tab.shape, len(TAB_FEATURES)


((12967, 18), (3242, 18), (5404, 18), 18)

In [10]:
tab_model = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

tab_model.fit(X_train_tab, y_train)

val_pred_tab = tab_model.predict(X_val_tab)

mse_tab = mean_squared_error(y_val, val_pred_tab)  # mean_squared_error exists across versions. [web:118]
rmse_tab = float(np.sqrt(mse_tab))
r2_tab = r2_score(y_val, val_pred_tab)

print({"tabular_rmse": rmse_tab, "tabular_r2": r2_tab})


{'tabular_rmse': 113737.7418450006, 'tabular_r2': 0.8969127535820007}


In [11]:
importances = tab_model.feature_importances_
imp_df = pd.DataFrame({"feature": TAB_FEATURES, "importance": importances}).sort_values("importance", ascending=False)
imp_df.head(25)


,feature,importance
8,grade,0.388662
5,waterfront,0.291289
2,sqft_living,0.073959
6,view,0.047742
14,lat,0.044541
12,yr_renovated,0.023966
1,bathrooms,0.023798
15,long,0.023347
11,yr_built,0.014379
16,sqft_living15,0.013157


In [12]:
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

test_pred_tab = tab_model.predict(X_test_tab)
pd.DataFrame({"id": test_ids, "predicted_price": test_pred_tab}).to_csv(out_dir / "predictions_tabular.csv", index=False)

out_dir / "predictions_tabular.csv"


PosixPath('outputs/predictions_tabular.csv')

In [13]:
class MultiModalDS(Dataset):
    def __init__(self, ids, X_tab, y=None, images_dir="data/images/train", transform=None):
        self.ids = ids
        self.X_tab = X_tab
        self.y = y
        self.images_dir = Path(images_dir)
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        pid = self.ids[i]
        img_path = self.images_dir / f"{pid}.png"
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        tab = torch.from_numpy(self.X_tab[i])  # float32
        if self.y is None:
            return img, tab, pid
        y = torch.tensor(float(self.y[i]), dtype=torch.float32)
        return img, tab, y, pid


In [14]:
class MultiModalRegressor(nn.Module):
    def __init__(self, tab_dim: int):
        super().__init__()

        weights = ResNet18_Weights.DEFAULT  # documented weights enum. [web:41]
        backbone = models.resnet18(weights=weights)  # resnet18 supports weights parameter. [web:41]

        self.cnn = nn.Sequential(*list(backbone.children())[:-1])  # pooled feature vector
        self.img_dim = 512

        self.tab_mlp = nn.Sequential(
            nn.Linear(tab_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.1),
        )

        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 128, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )

    def forward(self, img, tab):
        img_feat = self.cnn(img).flatten(1)
        tab_feat = self.tab_mlp(tab)
        fused = torch.cat([img_feat, tab_feat], dim=1)
        return self.head(fused).squeeze(1)


In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"

weights = ResNet18_Weights.DEFAULT
img_transform = weights.transforms()  # recommended preprocessing for these weights. [web:41]

ds_train = MultiModalDS(train_ids, X_train_tab, y=y_train, images_dir="data/images/train", transform=img_transform)
ds_val   = MultiModalDS(val_ids,   X_val_tab,   y=y_val,   images_dir="data/images/train", transform=img_transform)
ds_test  = MultiModalDS(test_ids,  X_test_tab,  y=None,   images_dir="data/images/test",  transform=img_transform)

dl_train = DataLoader(ds_train, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
dl_val   = DataLoader(ds_val,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
dl_test  = DataLoader(ds_test,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)


In [17]:
model = MultiModalRegressor(tab_dim=X_train_tab.shape[1]).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
loss_fn = nn.MSELoss()

best_rmse = float("inf")
(out_dir / "checkpoints").mkdir(parents=True, exist_ok=True)
best_path = out_dir / "checkpoints" / "mm_best.pt"

EPOCHS = 5

for epoch in range(1, EPOCHS + 1):
    model.train()
    for img, tab, y, _ in dl_train:
        img = img.to(device, non_blocking=True)
        tab = tab.to(device, non_blocking=True)
        y   = y.to(device, non_blocking=True)

        pred = model(img, tab)
        loss = loss_fn(pred, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for img, tab, y, _ in dl_val:
            img = img.to(device, non_blocking=True)
            tab = tab.to(device, non_blocking=True)
            pred = model(img, tab).detach().cpu().numpy()
            y_pred.extend(pred.tolist())
            y_true.extend(y.numpy().tolist())

    mse_mm = mean_squared_error(y_true, y_pred)  # mean_squared_error exists across versions. [web:118]
    rmse_mm = float(np.sqrt(mse_mm))
    r2_mm = r2_score(y_true, y_pred)

    print({
        "epoch": epoch,
        "tabular_rmse": rmse_tab,
        "multimodal_rmse": rmse_mm,
        "tabular_r2": r2_tab,
        "multimodal_r2": r2_mm,
    })

    if rmse_mm < best_rmse:
        best_rmse = rmse_mm
        torch.save(model.state_dict(), best_path)
        print("saved", best_path)


{'epoch': 1, 'tabular_rmse': 113737.7418450006, 'multimodal_rmse': 637050.093904901, 'tabular_r2': 0.8969127535820007, 'multimodal_r2': -2.234021168708881}
saved outputs/checkpoints/mm_best.pt


/home/omdeore/CDC_project/cdc_data_proj_venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'epoch': 2, 'tabular_rmse': 113737.7418450006, 'multimodal_rmse': 609112.4909543641, 'tabular_r2': 0.8969127535820007, 'multimodal_r2': -1.9565872611678277}
saved outputs/checkpoints/mm_best.pt


/home/omdeore/CDC_project/cdc_data_proj_venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'epoch': 3, 'tabular_rmse': 113737.7418450006, 'multimodal_rmse': 514703.9606734202, 'tabular_r2': 0.8969127535820007, 'multimodal_r2': -1.111109259223984}
saved outputs/checkpoints/mm_best.pt


/home/omdeore/CDC_project/cdc_data_proj_venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'epoch': 4, 'tabular_rmse': 113737.7418450006, 'multimodal_rmse': 315268.2449620635, 'tabular_r2': 0.8969127535820007, 'multimodal_r2': 0.20794352536610372}
saved outputs/checkpoints/mm_best.pt


/home/omdeore/CDC_project/cdc_data_proj_venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'epoch': 5, 'tabular_rmse': 113737.7418450006, 'multimodal_rmse': 307924.66183486313, 'tabular_r2': 0.8969127535820007, 'multimodal_r2': 0.24441272288434435}
saved outputs/checkpoints/mm_best.pt


In [19]:
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

preds = []
ids_out = []
with torch.no_grad():
    for img, tab, pid in dl_test:
        img = img.to(device, non_blocking=True)
        tab = tab.to(device, non_blocking=True)
        p = model(img, tab).detach().cpu().numpy()
        preds.extend(p.tolist())
        ids_out.extend(pid.tolist())

final = pd.DataFrame({"id": ids_out, "predicted_price": preds})
final.to_csv(out_dir / "predictions.csv", index=False)

final.head(), out_dir / "predictions.csv"

AttributeError: 'tuple' object has no attribute 'tolist'